# AI-Assisted Diagnosis of Cutaneous Melanoma
## Final Attention U-Net + EfficientNetB0 Google Colab Pipeline

**Project architecture**

Dermoscopic image  
→ preprocessing  
→ **Attention U-Net lesion segmentation**  
→ predicted binary lesion mask  
→ lesion ROI  
→ **EfficientNetB0 transfer-learning classifier**  
→ benign/malignant model probability  
→ confusion matrix + sensitivity/specificity/TP/TN/FP/FN  
→ Grad-CAM

### Dataset design used in this version
This notebook uses **300 matched ISIC 2016 Task 3B cases**. Every selected case has:

- a dermoscopic RGB image,
- an expert lesion segmentation mask,
- and a benign/malignant label.

The 300 cases are split as:

- **200 modelling cases**
  - 160 training
  - 40 validation
- **100 final testing cases**

The 100 final test cases remain excluded from model training, optimisation and model selection.

### Important
This is a research/educational prototype. It is **not a medical diagnostic device** and must not replace assessment by a qualified healthcare professional.


## 0. How to run this notebook

1. In Google Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run the notebook from top to bottom.
3. The official ISIC 2016 Task 3B ZIP is downloaded to temporary Colab storage (`/content`) only.
4. Only the selected 300 image/mask pairs are extracted.
5. TensorFlow `tf.data` reads images in batches during training rather than loading the whole dataset into RAM.
6. Google Drive is optional and is used only for small persistent outputs such as models, split CSVs and result tables.


## 1. Environment setup

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gradio>=4.0"]
    )

print("Environment setup complete.")

## 2. Imports and reproducibility

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Global configuration

In [ ]:
TOTAL_CASES = 300
MODELLING_CASES = 200
FINAL_TEST_CASES = 100
MODEL_TRAIN_CASES = 160
MODEL_VAL_CASES = 40

SEG_IMAGE_SIZE = (256, 256)
CLS_IMAGE_SIZE = (224, 224)

SEG_BATCH_SIZE = 4
CLS_BATCH_SIZE = 16

SEG_EPOCHS = 40
CLS_HEAD_EPOCHS = 15
CLS_FINE_TUNE_EPOCHS = 10

SEG_LEARNING_RATE = 1e-3
CLS_HEAD_LR = 1e-3
CLS_FINE_TUNE_LR = 1e-5

SEG_THRESHOLD = 0.50
CLS_THRESHOLD = 0.50

DO_FINE_TUNING = True

# Large dataset files stay only in /content.
RUNTIME_ROOT = Path("/content/melanoma_300_project")
DOWNLOAD_DIR = RUNTIME_ROOT / "downloads"
DATA_DIR = RUNTIME_ROOT / "selected_300"
ROI_DIR = RUNTIME_ROOT / "roi_images"

# Set False if you do not want Google Drive mounted.
SAVE_RESULTS_TO_DRIVE = True

for folder in [RUNTIME_ROOT, DOWNLOAD_DIR, DATA_DIR, ROI_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Runtime root:", RUNTIME_ROOT)
print("Only 300 selected cases will be extracted.")
print("Modelling: 160 train + 40 validation")
print("Final held-out test: 100")


## 4. Optional Google Drive output folder

In [ ]:
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        OUTPUT_DIR = Path(
            "/content/drive/MyDrive/Melanoma_AI_AttentionUNet_300_Case_Results"
        )
    except Exception as exc:
        print("Drive mount unavailable. Falling back to /content.")
        print("Reason:", exc)
        OUTPUT_DIR = RUNTIME_ROOT / "results"
else:
    OUTPUT_DIR = RUNTIME_ROOT / "results"

MODELS_DIR = OUTPUT_DIR / "models"
RESULTS_DIR = OUTPUT_DIR / "results"
SPLITS_DIR = OUTPUT_DIR / "splits"

for folder in [OUTPUT_DIR, MODELS_DIR, RESULTS_DIR, SPLITS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", OUTPUT_DIR)

## 5. Official ISIC 2016 Task 3B data

The official **ISIC 2016 Task 3B: Segmented Lesion Classification** training data contains:

- 900 dermoscopic JPEG images,
- 900 matching expert segmentation masks,
- 900 benign/malignant labels.

This notebook downloads the official training ZIP and CSV into temporary Colab storage, selects a stratified 300-case subset, extracts only those selected cases, and deletes the large ZIP afterwards.

In [ ]:
DATA_ZIP_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_Data.zip"
)

LABEL_CSV_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"
)

DATA_ZIP = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_Data.zip"
LABEL_CSV = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"


def download_file(url, destination, chunk_mb=8):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 100:
        print("Already downloaded:", destination.name)
        return destination

    tmp = destination.with_suffix(destination.suffix + ".part")
    start = tmp.stat().st_size if tmp.exists() else 0
    headers = {"Range": f"bytes={start}-"} if start else {}

    print("Downloading:", destination.name)

    with requests.get(
        url,
        stream=True,
        headers=headers,
        timeout=(30, 120),
    ) as response:
        response.raise_for_status()

        if start and response.status_code != 206:
            start = 0
            tmp.unlink(missing_ok=True)

        mode = "ab" if start else "wb"

        with open(tmp, mode) as handle:
            downloaded = start
            last_print = time.time()

            for chunk in response.iter_content(
                chunk_size=chunk_mb * 1024 * 1024
            ):
                if not chunk:
                    continue

                handle.write(chunk)
                downloaded += len(chunk)

                if time.time() - last_print > 5:
                    print(
                        f"  downloaded {downloaded / (1024**2):.1f} MB"
                    )
                    last_print = time.time()

    tmp.replace(destination)
    print("Finished:", destination)
    return destination


download_file(LABEL_CSV_URL, LABEL_CSV)
download_file(DATA_ZIP_URL, DATA_ZIP)

assert LABEL_CSV.exists(), "Ground-truth CSV download failed."
assert DATA_ZIP.exists(), "Training ZIP download failed."
assert zipfile.is_zipfile(DATA_ZIP), (
    "Downloaded training file is not a valid ZIP."
)

print("Official ISIC files are ready in temporary Colab storage.")

## 6. Load all labels and select exactly 300 cases

In [ ]:
labels_df = pd.read_csv(
    LABEL_CSV,
    header=None,
    names=["image_id", "diagnosis"],
)

labels_df["diagnosis"] = (
    labels_df["diagnosis"]
    .astype(str)
    .str.strip()
    .str.lower()
)

labels_df["label"] = labels_df["diagnosis"].map(
    {"benign": 0, "malignant": 1}
)

assert len(labels_df) == 900, (
    f"Expected 900 labels, found {len(labels_df)}."
)
assert labels_df["label"].notna().all(), (
    "Unexpected diagnosis values were found."
)
assert labels_df["image_id"].is_unique, (
    "Duplicate image IDs were found in the label CSV."
)

print("Full ISIC 2016 Task 3B label distribution:")
display(
    labels_df["diagnosis"]
    .value_counts()
    .to_frame("count")
)

selected_cases, unused = train_test_split(
    labels_df,
    train_size=TOTAL_CASES,
    stratify=labels_df["label"],
    random_state=SEED,
)

selected_cases = selected_cases.reset_index(drop=True)

assert len(selected_cases) == TOTAL_CASES

print(f"\nSelected {TOTAL_CASES}-case distribution:")
display(
    selected_cases["diagnosis"]
    .value_counts()
    .to_frame("count")
)


## 7. Create the 200 modelling / 100 testing split

In [ ]:
model_df, test_df = train_test_split(
    selected_cases,
    train_size=MODELLING_CASES,
    test_size=FINAL_TEST_CASES,
    stratify=selected_cases["label"],
    random_state=SEED,
)

train_df, val_df = train_test_split(
    model_df,
    train_size=MODEL_TRAIN_CASES,
    test_size=MODEL_VAL_CASES,
    stratify=model_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
model_df = model_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(model_df) == MODELLING_CASES
assert len(test_df) == FINAL_TEST_CASES
assert len(train_df) == MODEL_TRAIN_CASES
assert len(val_df) == MODEL_VAL_CASES

assert set(model_df.image_id).isdisjoint(set(test_df.image_id))
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))


def show_split(name, frame):
    counts = frame["diagnosis"].value_counts()
    percentages = (
        frame["diagnosis"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )

    print(f"\n{name}: n={len(frame)}")
    display(
        pd.DataFrame(
            {
                "count": counts,
                "percent": percentages,
            }
        )
    )


show_split("TRAIN", train_df)
show_split("VALIDATION", val_df)
show_split("FINAL TEST", test_df)

train_df[
    ["image_id", "diagnosis", "label"]
].to_csv(
    SPLITS_DIR / f"train_{MODEL_TRAIN_CASES}.csv",
    index=False,
)

val_df[
    ["image_id", "diagnosis", "label"]
].to_csv(
    SPLITS_DIR / f"validation_{MODEL_VAL_CASES}.csv",
    index=False,
)

test_df[
    ["image_id", "diagnosis", "label"]
].to_csv(
    SPLITS_DIR / f"final_test_{FINAL_TEST_CASES}.csv",
    index=False,
)

print("Split files saved to:", SPLITS_DIR)


## 8. Extract only the selected 300 images and masks

The 900-case ZIP is **not** extracted in full. Only 300 selected images and 300 matching masks are extracted.

After successful extraction, the large ZIP is deleted from temporary Colab storage.

In [ ]:
selected_ids = set(selected_cases["image_id"])

with zipfile.ZipFile(DATA_ZIP, "r") as archive:
    file_members = [
        member
        for member in archive.namelist()
        if not member.endswith("/")
    ]

    by_basename = {
        Path(member).name: member
        for member in file_members
    }

    missing_images = []
    missing_masks = []
    extracted_records = []

    for image_id in sorted(selected_ids):
        image_base = f"{image_id}.jpg"
        mask_base = f"{image_id}_Segmentation.png"

        image_member = by_basename.get(image_base)
        mask_member = by_basename.get(mask_base)

        if image_member is None:
            missing_images.append(image_id)
            continue

        if mask_member is None:
            missing_masks.append(image_id)
            continue

        image_output = DATA_DIR / image_base
        mask_output = DATA_DIR / mask_base

        if not image_output.exists():
            with archive.open(image_member) as src:
                with open(image_output, "wb") as dst_file:
                    shutil.copyfileobj(src, dst_file)

        if not mask_output.exists():
            with archive.open(mask_member) as src:
                with open(mask_output, "wb") as dst_file:
                    shutil.copyfileobj(src, dst_file)

        extracted_records.append(
            {
                "image_id": image_id,
                "image_path": str(image_output),
                "mask_path": str(mask_output),
            }
        )

assert not missing_images, (
    f"Missing selected images: {missing_images[:10]}"
)
assert not missing_masks, (
    f"Missing selected masks: {missing_masks[:10]}"
)
assert len(extracted_records) == TOTAL_CASES, (
    f"Expected {TOTAL_CASES} extracted pairs, found "
    f"{len(extracted_records)}."
)

path_df = pd.DataFrame(extracted_records)


def add_paths(frame):
    result = frame.merge(
        path_df,
        on="image_id",
        how="left",
    )
    assert result["image_path"].notna().all()
    assert result["mask_path"].notna().all()
    return result


train_df = add_paths(train_df)
val_df = add_paths(val_df)
model_df = add_paths(model_df)
test_df = add_paths(test_df)
selected_cases = add_paths(selected_cases)

if DATA_ZIP.exists():
    DATA_ZIP.unlink()
    print(
        "Deleted large temporary ZIP:",
        DATA_ZIP.name,
    )

print(
    "Extracted selected image-mask pairs:",
    len(selected_cases),
)
print("Selected data directory:", DATA_DIR)


## 9. Dataset sanity checks and visualisation

In [ ]:
def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_mask(path):
    with Image.open(path) as image:
        return image.convert("L").copy()


sample_rows = pd.concat(
    [
        train_df[train_df["label"] == 0].head(2),
        train_df[train_df["label"] == 1].head(2),
    ]
).reset_index(drop=True)

fig, axes = plt.subplots(
    len(sample_rows),
    3,
    figsize=(12, 4 * len(sample_rows)),
)

if len(sample_rows) == 1:
    axes = np.array([axes])

for row_index, row in sample_rows.iterrows():
    image = load_rgb(row["image_path"])
    mask = load_mask(row["mask_path"])

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(
        f"{row['image_id']} | {row['diagnosis']}"
    )
    axes[row_index, 0].axis("off")

    axes[row_index, 1].imshow(mask, cmap="gray")
    axes[row_index, 1].set_title("Expert mask")
    axes[row_index, 1].axis("off")

    axes[row_index, 2].imshow(image)
    axes[row_index, 2].imshow(
        mask,
        cmap="Reds",
        alpha=0.30,
    )
    axes[row_index, 2].set_title("Mask overlay")
    axes[row_index, 2].axis("off")

plt.tight_layout()
plt.show()

### Advanced Preprocessing: Hair Removal (DullRazor)
One of the most effective ways to improve segmentation in dermoscopy is to remove hair artifacts that the model might mistake for lesion boundaries.

In [ ]:
import cv2

def dull_razor(img_pil):
    # Convert PIL to OpenCV format
    img = np.array(img_pil)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Black hat filtering to identify hair
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

    # Intensify hair pixels
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)

    # Inpaint the original image using the mask
    dst = cv2.inpaint(img, mask, 1, cv2.INPAINT_TELEA)

    return Image.fromarray(dst)

# Example usage on the first train image
original_sample = load_rgb(train_df.iloc[0]['image_path'])
cleaned_sample = dull_razor(original_sample)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(original_sample); ax[0].set_title('Original')
ax[1].imshow(cleaned_sample); ax[1].set_title('After DullRazor')
plt.show()

### 9.1 Visual Inspection of DullRazor
We verify that the `dull_razor` function handles various cases (hair, no hair, dark lesions) without distorting the lesion anatomy.

In [ ]:
sample_indices = [0, 5, 10, 15, 20, 25] # Covering diverse samples
fig, axes = plt.subplots(len(sample_indices), 2, figsize=(10, 5 * len(sample_indices)))

for i, idx in enumerate(sample_indices):
    row = train_df.iloc[idx]
    img = load_rgb(row['image_path'])
    cleaned = dull_razor(img)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Original: {row['image_id']}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(cleaned)
    axes[i, 1].set_title("DullRazor Applied")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

### 9.2 Standardized Mask Visualization
We define a reusable overlay function to ensure all mask comparisons are fair and consistent across baseline, DullRazor, and ground truth.

In [ ]:
def segmentation_overlay(image_pil, mask_binary, alpha=0.30):
    """Standardized red-overlay for fair visual comparison."""
    img = image_pil.convert("RGB")
    img_arr = np.array(img).astype(np.float32)

    # Ensure mask is binary and resized to match image using nearest neighbor
    mask_pil = Image.fromarray((mask_binary * 255).astype(np.uint8))
    mask_resized = np.array(mask_pil.resize(img.size, Image.Resampling.NEAREST)) > 127

    overlay = img_arr.copy()
    # Apply red tint to the Red channel where mask is True
    overlay[mask_resized, 0] = np.clip(overlay[mask_resized, 0] * (1 - alpha) + 255 * alpha, 0, 255)
    return Image.fromarray(overlay.astype(np.uint8))

# Helper for custom preprocessing inference
def predict_mask_with_preprocessing(model, image_pil, preprocess_fn=None):
    proc_img = preprocess_fn(image_pil) if preprocess_fn else image_pil
    prob, binary = predict_mask_for_pil(model, proc_img)
    return prob, binary

# PART A — LESION SEGMENTATION

This version focuses only on **Attention U-Net**.

The model uses:

- 160 training cases,
- 40 validation cases,
- paired image/mask augmentation,
- Dice + IoU evaluation,
- early stopping,
- learning-rate reduction,
- and the final 100 cases remain untouched until held-out testing.

Attention gates are applied to the encoder-to-decoder skip connections to emphasise lesion-relevant features.


## 10. `tf.data` segmentation pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_segmentation_pair(
    image_path,
    mask_path,
):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )
    image = tf.image.resize(
        image,
        SEG_IMAGE_SIZE,
        method="bilinear",
    )
    image = tf.cast(image, tf.float32) / 255.0

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(
        mask_bytes,
        channels=1,
    )
    mask = tf.image.resize(
        mask,
        SEG_IMAGE_SIZE,
        method="nearest",
    )
    mask = tf.cast(mask > 127, tf.float32)

    return image, mask


def paired_segmentation_augmentation(
    image,
    mask,
):
    seed1 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )
    seed2 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )

    image = tf.image.stateless_random_flip_left_right(
        image,
        seed1,
    )
    mask = tf.image.stateless_random_flip_left_right(
        mask,
        seed1,
    )

    image = tf.image.stateless_random_flip_up_down(
        image,
        seed2,
    )
    mask = tf.image.stateless_random_flip_up_down(
        mask,
        seed2,
    )

    image = tf.image.random_brightness(
        image,
        max_delta=0.06,
    )
    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10,
    )
    image = tf.clip_by_value(
        image,
        0.0,
        1.0,
    )

    return image, mask


def make_seg_dataset(
    frame,
    training=False,
):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["image_path"].astype(str).values,
            frame["mask_path"].astype(str).values,
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_segmentation_pair,
        num_parallel_calls=AUTOTUNE,
    )

    if training:
        ds = ds.map(
            paired_segmentation_augmentation,
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(SEG_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_seg_ds = make_seg_dataset(
    train_df,
    training=True,
)
val_seg_ds = make_seg_dataset(
    val_df,
    training=False,
)
test_seg_ds = make_seg_dataset(
    test_df,
    training=False,
)

images_batch, masks_batch = next(
    iter(train_seg_ds)
)

print(
    "Image batch shape:",
    images_batch.shape,
)
print(
    "Mask batch shape:",
    masks_batch.shape,
)
print(
    "Image range:",
    float(tf.reduce_min(images_batch)),
    "to",
    float(tf.reduce_max(images_batch)),
)
print(
    "Mask values:",
    np.unique(masks_batch.numpy()),
)

## 11. Dice, IoU/Jaccard and segmentation loss

In [ ]:
EPS = 1e-7


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def dice_coefficient(y_true, y_pred):
    y_true = tf.cast(
        y_true,
        tf.float32,
    )
    y_pred = tf.cast(
        y_pred,
        tf.float32,
    )

    y_true_f = tf.reshape(
        y_true,
        [-1],
    )
    y_pred_f = tf.reshape(
        y_pred,
        [-1],
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    return (
        2.0 * intersection + EPS
    ) / (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        + EPS
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(
        y_true,
        y_pred,
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(
        y_true > 0.5,
        tf.float32,
    )
    y_pred = tf.cast(
        y_pred > SEG_THRESHOLD,
        tf.float32,
    )

    y_true_f = tf.reshape(
        y_true,
        [-1],
    )
    y_pred_f = tf.reshape(
        y_pred,
        [-1],
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    union = (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        - intersection
    )

    return (
        intersection + EPS
    ) / (
        union + EPS
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def combined_segmentation_loss(
    y_true,
    y_pred,
):
    bce = tf.reduce_mean(
        keras.losses.binary_crossentropy(
            y_true,
            y_pred,
        )
    )

    return (
        bce
        + dice_loss(y_true, y_pred)
    )


_test_true = tf.constant(
    [[[[1.0]], [[0.0]]]]
)
_test_pred = tf.constant(
    [[[[1.0]], [[0.0]]]]
)

print(
    "Perfect Dice:",
    float(
        dice_coefficient(
            _test_true,
            _test_pred,
        )
    ),
)

print(
    "Perfect IoU:",
    float(
        iou_coefficient(
            _test_true,
            _test_pred,
        )
    ),
)

## 12. Attention U-Net architecture


In [ ]:
def conv_block(
    x,
    filters,
    name,
):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv1",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu1",
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv2",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu2",
    )(x)

    return x


def attention_gate(
    skip,
    gating,
    inter_filters,
    name,
):
    theta = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_theta",
    )(skip)

    phi = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_phi",
    )(gating)

    x = layers.Add(
        name=f"{name}_add"
    )([theta, phi])

    x = layers.Activation(
        "relu",
        name=f"{name}_relu",
    )(x)

    psi = layers.Conv2D(
        1,
        1,
        padding="same",
        activation="sigmoid",
        name=f"{name}_psi",
    )(x)

    return layers.Multiply(
        name=f"{name}_weighted_skip"
    )([skip, psi])


def build_attention_unet(
    input_shape=(256, 256, 3),
):
    inputs = keras.Input(
        shape=input_shape,
        name="image",
    )

    c1 = conv_block(inputs, 32, "enc1")
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64, "enc2")
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128, "enc3")
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256, "enc4")
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(
        p4,
        512,
        "bottleneck",
    )

    def decoder_step(
        x,
        skip,
        filters,
        level,
    ):
        x = layers.Conv2DTranspose(
            filters,
            2,
            strides=2,
            padding="same",
            name=f"up{level}",
        )(x)

        skip = attention_gate(
            skip,
            x,
            max(filters // 2, 1),
            name=f"attention{level}",
        )

        x = layers.Concatenate(
            name=f"concat{level}"
        )([x, skip])

        return conv_block(
            x,
            filters,
            f"dec{level}",
        )

    d4 = decoder_step(bottleneck, c4, 256, 4)
    d3 = decoder_step(d4, c3, 128, 3)
    d2 = decoder_step(d3, c2, 64, 2)
    d1 = decoder_step(d2, c1, 32, 1)

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        dtype="float32",
        name="segmentation_mask",
    )(d1)

    return keras.Model(
        inputs,
        outputs,
        name="Attention_U_Net",
    )


attention_unet = build_attention_unet(
    input_shape=(
        *SEG_IMAGE_SIZE,
        3,
    )
)

print(
    "Attention U-Net parameters:",
    attention_unet.count_params(),
)

dummy = tf.zeros(
    (1, *SEG_IMAGE_SIZE, 3),
    dtype=tf.float32,
)

assert attention_unet(dummy).shape == (
    1,
    SEG_IMAGE_SIZE[0],
    SEG_IMAGE_SIZE[1],
    1,
)

print(
    "Attention U-Net passed "
    "the synthetic shape test."
)


## 13. Train Attention U-Net


In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "dice_loss": dice_loss,
    "iou_coefficient": iou_coefficient,
    "combined_segmentation_loss": (
        combined_segmentation_loss
    ),
}


def compile_segmentation_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=SEG_LEARNING_RATE
        ),
        loss=combined_segmentation_loss,
        metrics=[
            dice_coefficient,
            iou_coefficient,
        ],
    )


def segmentation_callbacks(model_path):
    return [
        keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

attention_model = build_attention_unet(
    input_shape=(
        *SEG_IMAGE_SIZE,
        3,
    )
)

compile_segmentation_model(
    attention_model
)

ATTENTION_MODEL_PATH = (
    MODELS_DIR
    / "attention_unet_best.keras"
)

start_time = time.time()

attention_fit = attention_model.fit(
    train_seg_ds,
    validation_data=val_seg_ds,
    epochs=SEG_EPOCHS,
    callbacks=segmentation_callbacks(
        ATTENTION_MODEL_PATH
    ),
    verbose=1,
)

attention_training_seconds = (
    time.time() - start_time
)

attention_history = (
    attention_fit.history
)

selected_segmenter = (
    keras.models.load_model(
        ATTENTION_MODEL_PATH,
        custom_objects=CUSTOM_OBJECTS,
    )
)

attention_validation = (
    selected_segmenter.evaluate(
        val_seg_ds,
        verbose=0,
        return_dict=True,
    )
)

SELECTED_SEGMENTER_NAME = (
    "Attention U-Net"
)

SELECTED_SEGMENTER_PATH = (
    str(ATTENTION_MODEL_PATH)
)

print("\nAttention U-Net training finished.")
print(
    "Best checkpoint:",
    ATTENTION_MODEL_PATH,
)
print(
    "Validation Dice:",
    attention_validation[
        "dice_coefficient"
    ],
)
print(
    "Validation IoU:",
    attention_validation[
        "iou_coefficient"
    ],
)
print(
    "Parameters:",
    selected_segmenter.count_params(),
)


## 14. Lock the best Attention U-Net checkpoint


In [ ]:
best_validation_dice = float(
    np.max(
        attention_history[
            "val_dice_coefficient"
        ]
    )
)

best_validation_iou = float(
    np.max(
        attention_history[
            "val_iou_coefficient"
        ]
    )
)

attention_validation_summary = pd.DataFrame(
    {
        "Metric": [
            "Best Validation Dice",
            "Best Validation IoU",
            "Validation Loss (best checkpoint)",
            "Parameters",
            "Training time (min)",
        ],
        "Value": [
            best_validation_dice,
            best_validation_iou,
            float(attention_validation["loss"]),
            int(selected_segmenter.count_params()),
            attention_training_seconds / 60.0,
        ],
    }
)

display(
    attention_validation_summary
)

attention_validation_summary.to_csv(
    RESULTS_DIR
    / "attention_unet_validation_summary.csv",
    index=False,
)

print(
    "Attention U-Net checkpoint locked "
    "using validation performance only."
)


## 15. Attention U-Net training curves


In [ ]:
history = attention_history
epochs = range(
    1,
    len(history["loss"]) + 1,
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
)

axes[0].plot(
    epochs,
    history["loss"],
    label="Train",
)
axes[0].plot(
    epochs,
    history["val_loss"],
    linestyle="--",
    label="Validation",
)
axes[0].set_title(
    "Attention U-Net Loss"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(
    epochs,
    history["dice_coefficient"],
    label="Train",
)
axes[1].plot(
    epochs,
    history["val_dice_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[1].set_title(
    "Attention U-Net Dice"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()

axes[2].plot(
    epochs,
    history["iou_coefficient"],
    label="Train",
)
axes[2].plot(
    epochs,
    history["val_iou_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[2].set_title(
    "Attention U-Net IoU / Jaccard"
)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].legend()

plt.tight_layout()
plt.show()


# PART B — LESION ROI GENERATION

The selected segmentation model is now locked.

The **predicted mask** is used to create the ROI supplied to EfficientNetB0 so the classification stage matches the intended end-to-end pipeline.

## 16. Predicted-mask and ROI extraction helpers

In [ ]:
def predict_mask_for_pil(
    model,
    image_pil,
):
    image_resized = image_pil.resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0],
        ),
        Image.Resampling.BILINEAR,
    )

    x = (
        np.asarray(image_resized)
        .astype("float32")
        / 255.0
    )

    prediction = model.predict(
        x[None, ...],
        verbose=0,
    )[0, ..., 0]

    binary = (
        prediction >= SEG_THRESHOLD
    ).astype(np.uint8)

    return prediction, binary


def extract_roi_from_binary_mask(
    image_pil,
    binary_mask,
    padding_fraction=0.10,
):
    """
    Extract a padded lesion crop.
    Falls back to the full image if
    the predicted mask is empty or invalid.
    """

    image_array = np.asarray(
        image_pil.convert("RGB")
    )

    original_h, original_w = (
        image_array.shape[:2]
    )

    mask_pil = Image.fromarray(
        (
            binary_mask * 255
        ).astype(np.uint8)
    )

    mask_full = mask_pil.resize(
        (
            original_w,
            original_h,
        ),
        Image.Resampling.NEAREST,
    )

    mask_full = (
        np.asarray(mask_full)
        > 0
    )

    ys, xs = np.where(
        mask_full
    )

    if (
        len(xs) == 0
        or len(ys) == 0
    ):
        return (
            image_pil.copy(),
            (
                0,
                0,
                original_w,
                original_h,
            ),
            True,
        )

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    box_w = max(
        1,
        x_max - x_min + 1,
    )
    box_h = max(
        1,
        y_max - y_min + 1,
    )

    pad_x = int(
        round(
            box_w
            * padding_fraction
        )
    )

    pad_y = int(
        round(
            box_h
            * padding_fraction
        )
    )

    x0 = max(
        0,
        x_min - pad_x,
    )
    y0 = max(
        0,
        y_min - pad_y,
    )
    x1 = min(
        original_w,
        x_max + pad_x + 1,
    )
    y1 = min(
        original_h,
        y_max + pad_y + 1,
    )

    if (
        (x1 - x0) < 8
        or (y1 - y0) < 8
    ):
        return (
            image_pil.copy(),
            (
                0,
                0,
                original_w,
                original_h,
            ),
            True,
        )

    roi = image_pil.crop(
        (
            x0,
            y0,
            x1,
            y1,
        )
    )

    return (
        roi,
        (
            x0,
            y0,
            x1,
            y1,
        ),
        False,
    )


synthetic_image = Image.new(
    "RGB",
    (100, 80),
    "white",
)

empty_mask = np.zeros(
    (256, 256),
    dtype=np.uint8,
)

(
    roi_empty,
    box_empty,
    fallback_empty,
) = extract_roi_from_binary_mask(
    synthetic_image,
    empty_mask,
)

assert fallback_empty is True

normal_mask = np.zeros(
    (256, 256),
    dtype=np.uint8,
)
normal_mask[
    80:180,
    90:190,
] = 1

(
    roi_normal,
    box_normal,
    fallback_normal,
) = extract_roi_from_binary_mask(
    synthetic_image,
    normal_mask,
)

assert fallback_normal is False

print("ROI safety tests passed.")

## 16A. Validation-only DullRazor experiment

This experiment runs **after** Standard U-Net and Attention U-Net have been trained and the selected segmenter has been locked using validation performance.

It compares the locked segmenter on the **20 modelling-validation cases only** using:

- baseline input preprocessing, versus
- DullRazor hair removal at inference time.

**Important:** the final 100-case test set is not used here. This cell does not automatically change the final pipeline; it only reports whether DullRazor appears helpful on validation data.


In [ ]:
from scipy.stats import wilcoxon

# ------------------------------------------------------------
# Safety checks: this section MUST run only after model training
# and model selection.
# ------------------------------------------------------------
required_names = [
    "selected_segmenter",
    "SELECTED_SEGMENTER_NAME",
    "val_df",
    "predict_mask_for_pil",
    "dull_razor",
    "load_rgb",
    "load_mask",
    "SEG_IMAGE_SIZE",
    "SEG_THRESHOLD",
    "EPS",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the notebook from the beginning through model selection "
        "before this DullRazor experiment. Missing: "
        + ", ".join(missing)
    )


def predict_mask_with_preprocessing(model, image_pil, preprocess_fn=None):
    """Run segmentation after optional image-only preprocessing."""
    processed = preprocess_fn(image_pil) if preprocess_fn is not None else image_pil
    return predict_mask_for_pil(model, processed)


def segmentation_metrics_np(true_mask, pred_mask):
    """Binary mask metrics for one image."""
    true_mask = np.asarray(true_mask).astype(bool)
    pred_mask = np.asarray(pred_mask).astype(bool)

    tp = np.logical_and(true_mask, pred_mask).sum()
    tn = np.logical_and(~true_mask, ~pred_mask).sum()
    fp = np.logical_and(~true_mask, pred_mask).sum()
    fn = np.logical_and(true_mask, ~pred_mask).sum()

    true_sum = true_mask.sum()
    pred_sum = pred_mask.sum()
    union = np.logical_or(true_mask, pred_mask).sum()

    dice = (2.0 * tp + EPS) / (true_sum + pred_sum + EPS)
    iou = (tp + EPS) / (union + EPS)
    sensitivity = (tp + EPS) / (tp + fn + EPS)
    specificity = (tn + EPS) / (tn + fp + EPS)
    area_fraction = pred_sum / true_mask.size

    return {
        "dice": float(dice),
        "iou": float(iou),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "area_fraction": float(area_fraction),
    }


validation_rows = []

for _, row in val_df.iterrows():
    image_pil = load_rgb(row["image_path"])

    true_mask_pil = load_mask(row["mask_path"]).resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]),
        Image.Resampling.NEAREST,
    )
    true_mask = np.asarray(true_mask_pil) > 127

    # Baseline prediction
    base_prob, base_mask = predict_mask_with_preprocessing(
        selected_segmenter,
        image_pil,
        preprocess_fn=None,
    )

    # DullRazor prediction
    dr_prob, dr_mask = predict_mask_with_preprocessing(
        selected_segmenter,
        image_pil,
        preprocess_fn=dull_razor,
    )

    base_m = segmentation_metrics_np(true_mask, base_mask)
    dr_m = segmentation_metrics_np(true_mask, dr_mask)

    validation_rows.append(
        {
            "image_id": row["image_id"],
            "diagnosis": row["diagnosis"],
            "baseline_dice": base_m["dice"],
            "dullrazor_dice": dr_m["dice"],
            "dice_change": dr_m["dice"] - base_m["dice"],
            "baseline_iou": base_m["iou"],
            "dullrazor_iou": dr_m["iou"],
            "iou_change": dr_m["iou"] - base_m["iou"],
            "baseline_sensitivity": base_m["sensitivity"],
            "dullrazor_sensitivity": dr_m["sensitivity"],
            "baseline_specificity": base_m["specificity"],
            "dullrazor_specificity": dr_m["specificity"],
            "ground_truth_area": float(true_mask.mean()),
            "baseline_area": base_m["area_fraction"],
            "dullrazor_area": dr_m["area_fraction"],
        }
    )

dullrazor_validation_df = pd.DataFrame(validation_rows)

print(
    f"Validation-only DullRazor experiment using: "
    f"{SELECTED_SEGMENTER_NAME}"
)

summary = pd.DataFrame(
    {
        "Method": ["Baseline", "DullRazor"],
        "Mean Dice": [
            dullrazor_validation_df["baseline_dice"].mean(),
            dullrazor_validation_df["dullrazor_dice"].mean(),
        ],
        "Mean IoU": [
            dullrazor_validation_df["baseline_iou"].mean(),
            dullrazor_validation_df["dullrazor_iou"].mean(),
        ],
        "Mean Sensitivity": [
            dullrazor_validation_df["baseline_sensitivity"].mean(),
            dullrazor_validation_df["dullrazor_sensitivity"].mean(),
        ],
        "Mean Specificity": [
            dullrazor_validation_df["baseline_specificity"].mean(),
            dullrazor_validation_df["dullrazor_specificity"].mean(),
        ],
    }
)

display(summary.round(4))

dice_improved = int((dullrazor_validation_df["dice_change"] > 0.001).sum())
dice_worsened = int((dullrazor_validation_df["dice_change"] < -0.001).sum())
dice_unchanged = len(dullrazor_validation_df) - dice_improved - dice_worsened

iou_improved = int((dullrazor_validation_df["iou_change"] > 0.001).sum())
iou_worsened = int((dullrazor_validation_df["iou_change"] < -0.001).sum())
iou_unchanged = len(dullrazor_validation_df) - iou_improved - iou_worsened

mean_dice_change = dullrazor_validation_df["dice_change"].mean()
mean_iou_change = dullrazor_validation_df["iou_change"].mean()

print("\nCase-by-case comparison")
print(
    f"Dice — improved: {dice_improved}, "
    f"worsened: {dice_worsened}, unchanged: {dice_unchanged}"
)
print(
    f"IoU  — improved: {iou_improved}, "
    f"worsened: {iou_worsened}, unchanged: {iou_unchanged}"
)
print(f"Mean Dice change: {mean_dice_change:+.4f}")
print(f"Mean IoU change:  {mean_iou_change:+.4f}")

# Robust paired Wilcoxon tests.
def safe_wilcoxon(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = b - a

    if np.allclose(diff, 0):
        return np.nan, 1.0

    try:
        stat, p = wilcoxon(a, b, zero_method="wilcox")
        return float(stat), float(p)
    except ValueError:
        return np.nan, np.nan


dice_stat, dice_p = safe_wilcoxon(
    dullrazor_validation_df["baseline_dice"],
    dullrazor_validation_df["dullrazor_dice"],
)

iou_stat, iou_p = safe_wilcoxon(
    dullrazor_validation_df["baseline_iou"],
    dullrazor_validation_df["dullrazor_iou"],
)

print("\nPaired Wilcoxon signed-rank tests")
print(f"Dice p-value: {dice_p:.4f}" if np.isfinite(dice_p) else "Dice p-value: unavailable")
print(f"IoU  p-value: {iou_p:.4f}" if np.isfinite(iou_p) else "IoU p-value: unavailable")
print(f"Note: n={len(val_df)} validation cases; interpret statistical tests cautiously.")

# Validation-only recommendation. This does NOT change the actual final pipeline.
if (
    np.isfinite(dice_p)
    and dice_p < 0.05
    and mean_dice_change > 0.005
    and dice_improved > dice_worsened
):
    DULLRAZOR_RECOMMENDATION = "KEEP DULLRAZOR FOR FURTHER VALIDATION"
elif mean_dice_change > 0:
    DULLRAZOR_RECOMMENDATION = (
        "MIXED BENEFIT — DO NOT REPLACE BASELINE YET"
    )
else:
    DULLRAZOR_RECOMMENDATION = "DO NOT KEEP DULLRAZOR"

print("\n=== VALIDATION-ONLY RECOMMENDATION ===")
print(DULLRAZOR_RECOMMENDATION)

# Save validation results if results directory is available.
if "RESULTS_DIR" in globals():
    dullrazor_validation_df.to_csv(
        RESULTS_DIR / "dullrazor_validation_comparison.csv",
        index=False,
    )


### 16A.1 Visual comparison of the same validation cases

The ground-truth, baseline prediction and DullRazor prediction use the same red-overlay function so differences are not caused by plotting style.


In [ ]:
# Show the 3 biggest improvements and 3 biggest declines.
comparison_cases = pd.concat(
    [
        dullrazor_validation_df.nlargest(3, "dice_change"),
        dullrazor_validation_df.nsmallest(3, "dice_change"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(
    len(comparison_cases),
    5,
    figsize=(18, 4 * len(comparison_cases)),
)

if len(comparison_cases) == 1:
    axes = np.array([axes])

for plot_row, (_, result) in enumerate(comparison_cases.iterrows()):
    source_row = val_df.loc[
        val_df["image_id"] == result["image_id"]
    ].iloc[0]

    image_pil = load_rgb(source_row["image_path"])
    dr_image = dull_razor(image_pil)

    true_mask_pil = load_mask(source_row["mask_path"]).resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]),
        Image.Resampling.NEAREST,
    )
    true_mask = (np.asarray(true_mask_pil) > 127).astype(np.uint8)

    _, base_mask = predict_mask_with_preprocessing(
        selected_segmenter,
        image_pil,
        None,
    )
    _, dr_mask = predict_mask_with_preprocessing(
        selected_segmenter,
        image_pil,
        dull_razor,
    )

    axes[plot_row, 0].imshow(image_pil)
    axes[plot_row, 0].set_title(f"{result['image_id']}\nOriginal")

    axes[plot_row, 1].imshow(dr_image)
    axes[plot_row, 1].set_title("DullRazor")

    axes[plot_row, 2].imshow(
        segmentation_overlay(image_pil, true_mask)
    )
    axes[plot_row, 2].set_title("Expert overlay")

    axes[plot_row, 3].imshow(
        segmentation_overlay(image_pil, base_mask)
    )
    axes[plot_row, 3].set_title(
        f"Baseline\nDice {result['baseline_dice']:.3f}"
    )

    axes[plot_row, 4].imshow(
        segmentation_overlay(image_pil, dr_mask)
    )
    axes[plot_row, 4].set_title(
        f"DullRazor\nDice {result['dullrazor_dice']:.3f}"
    )

    for ax in axes[plot_row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 17. Generate ROI files for all selected cases

In [ ]:
ROI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

roi_records = []

for index, row in selected_cases.iterrows():
    image_pil = load_rgb(
        row["image_path"]
    )

    (
        probability_mask,
        binary_mask,
    ) = predict_mask_for_pil(
        selected_segmenter,
        image_pil,
    )

    (
        roi,
        bbox,
        fallback,
    ) = extract_roi_from_binary_mask(
        image_pil,
        binary_mask,
        padding_fraction=0.10,
    )

    roi_path = (
        ROI_DIR
        / f"{row['image_id']}_ROI.jpg"
    )

    roi.convert("RGB").save(
        roi_path,
        quality=95,
    )

    roi_records.append(
        {
            "image_id": row["image_id"],
            "roi_path": str(roi_path),
            "roi_fallback": bool(
                fallback
            ),
            "bbox": json.dumps(
                [
                    int(value)
                    for value
                    in bbox
                ]
            ),
        }
    )

    if (
        index + 1
    ) % 25 == 0:
        print(
            f"Generated "
            f"{index + 1}/200 ROIs"
        )

roi_df = pd.DataFrame(
    roi_records
)


def attach_rois(frame):
    result = frame.merge(
        roi_df,
        on="image_id",
        how="left",
    )

    assert (
        result["roi_path"]
        .notna()
        .all()
    )

    return result


train_df = attach_rois(
    train_df
)
val_df = attach_rois(
    val_df
)
model_df = attach_rois(
    model_df
)
test_df = attach_rois(
    test_df
)

print("ROI generation complete.")
print(
    "Fallback ROIs:",
    int(
        roi_df[
            "roi_fallback"
        ].sum()
    ),
    "out of 200",
)

## 18. Visualise original → predicted mask → ROI

In [ ]:
example = test_df.iloc[0]

image_pil = load_rgb(
    example["image_path"]
)

(
    probability_mask,
    binary_mask,
) = predict_mask_for_pil(
    selected_segmenter,
    image_pil,
)

roi_pil = load_rgb(
    example["roi_path"]
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4),
)

axes[0].imshow(
    image_pil
)
axes[0].set_title(
    "Original"
)
axes[0].axis(
    "off"
)

axes[1].imshow(
    binary_mask,
    cmap="gray",
)
axes[1].set_title(
    "Predicted lesion mask"
)
axes[1].axis(
    "off"
)

axes[2].imshow(
    roi_pil
)
axes[2].set_title(
    "Lesion ROI"
)
axes[2].axis(
    "off"
)

plt.tight_layout()
plt.show()

# PART C — EFFICIENTNETB0 CLASSIFICATION

EfficientNetB0 performs binary classification:

- `0 = benign`
- `1 = malignant`

Transfer learning is used because the modelling dataset is small.

## 19. `tf.data` classification pipeline

In [ ]:
classification_augmentation = (
    keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal_and_vertical",
                seed=SEED,
            ),
            layers.RandomRotation(
                0.05,
                seed=SEED,
            ),
            layers.RandomZoom(
                0.10,
                seed=SEED,
            ),
            layers.RandomContrast(
                0.10,
                seed=SEED,
            ),
        ],
        name=(
            "classification_augmentation"
        ),
    )
)


def decode_classification_image(
    path,
    label,
):
    image_bytes = tf.io.read_file(
        path
    )

    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )

    image = tf.image.resize(
        image,
        CLS_IMAGE_SIZE,
        method="bilinear",
    )

    # EfficientNetB0 in Keras contains
    # its own input Rescaling layer.
    # Keep pixels in the 0-255 range.
    image = tf.cast(
        image,
        tf.float32,
    )

    label = tf.cast(
        label,
        tf.float32,
    )

    return image, label


def make_cls_dataset(
    frame,
    training=False,
):
    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                frame[
                    "roi_path"
                ]
                .astype(str)
                .values,
                frame[
                    "label"
                ]
                .astype(
                    np.float32
                )
                .values,
            )
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_classification_image,
        num_parallel_calls=AUTOTUNE,
    )

    ds = ds.batch(
        CLS_BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_cls_ds = make_cls_dataset(
    train_df,
    training=True,
)

val_cls_ds = make_cls_dataset(
    val_df,
    training=False,
)

test_cls_ds = make_cls_dataset(
    test_df,
    training=False,
)

cls_images, cls_labels = next(
    iter(train_cls_ds)
)

print(
    "Classification image batch:",
    cls_images.shape,
)
print(
    "Pixel range:",
    float(
        tf.reduce_min(
            cls_images
        )
    ),
    "to",
    float(
        tf.reduce_max(
            cls_images
        )
    ),
)
print(
    "Labels:",
    cls_labels.numpy()[:10],
)

## 20. Class weights

In [ ]:
train_labels = (
    train_df["label"]
    .astype(int)
    .values
)

classes = np.unique(
    train_labels
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

CLASS_WEIGHTS = {
    int(class_value): float(weight)
    for class_value, weight
    in zip(classes, weights)
}

print(
    "Training class counts:"
)

display(
    train_df[
        "diagnosis"
    ]
    .value_counts()
    .to_frame("count")
)

print(
    "Class weights:",
    CLASS_WEIGHTS,
)

## 21. Build EfficientNetB0 transfer-learning classifier

In [ ]:
def build_efficientnet_classifier():
    inputs = keras.Input(
        shape=(
            *CLS_IMAGE_SIZE,
            3,
        ),
        name="roi_image",
    )

    x = (
        classification_augmentation(
            inputs
        )
    )

    backbone = (
        keras.applications
        .EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(
                *CLS_IMAGE_SIZE,
                3,
            ),
        )
    )

    backbone._name = (
        "efficientnetb0"
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False,
    )

    x = layers.GlobalAveragePooling2D(
        name="gap"
    )(x)

    x = layers.Dropout(
        0.30,
        name="dropout",
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="prediction",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name=(
            "Melanoma_EfficientNetB0"
        ),
    )


classifier = (
    build_efficientnet_classifier()
)

classifier.compile(
    optimizer=keras.optimizers.Adam(
        CLS_HEAD_LR
    ),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        keras.metrics.AUC(
            name="auc"
        ),
        keras.metrics.Precision(
            name="precision"
        ),
        keras.metrics.Recall(
            name="recall"
        ),
    ],
)

classifier.summary()

dummy_cls = tf.zeros(
    (
        1,
        224,
        224,
        3,
    ),
    dtype=tf.float32,
)

dummy_output = classifier(
    dummy_cls,
    training=False,
)

assert dummy_output.shape == (
    1,
    1,
)

print(
    "EfficientNetB0 classifier "
    "passed the synthetic shape test."
)

## 22. Train classification head

In [ ]:
HEAD_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_head_best.keras"
)

head_callbacks = [
    keras.callbacks.ModelCheckpoint(
        HEAD_MODEL_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

head_history = classifier.fit(
    train_cls_ds,
    validation_data=val_cls_ds,
    epochs=CLS_HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=head_callbacks,
    verbose=1,
)

## 23. Optional fine-tuning

In [ ]:
FINE_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_finetuned_best.keras"
)

fine_history = None

if DO_FINE_TUNING:
    classifier = keras.models.load_model(
        HEAD_MODEL_PATH
    )

    backbone = classifier.get_layer(
        "efficientnetb0"
    )

    backbone.trainable = True

    for layer in backbone.layers[:-20]:
        layer.trainable = False

    for layer in backbone.layers:
        if isinstance(
            layer,
            layers.BatchNormalization,
        ):
            layer.trainable = False

    classifier.compile(
        optimizer=keras.optimizers.Adam(
            CLS_FINE_TUNE_LR
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            keras.metrics.AUC(
                name="auc"
            ),
            keras.metrics.Precision(
                name="precision"
            ),
            keras.metrics.Recall(
                name="recall"
            ),
        ],
    )

    fine_callbacks = [
        keras.callbacks.ModelCheckpoint(
            FINE_MODEL_PATH,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    fine_history = classifier.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_FINE_TUNE_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=fine_callbacks,
        verbose=1,
    )
else:
    print(
        "Fine-tuning disabled."
    )

## 24. Select the classifier using validation ROC-AUC

In [ ]:
def predict_probabilities(
    model,
    dataset,
):
    return (
        model.predict(
            dataset,
            verbose=0,
        )
        .reshape(-1)
    )


head_model = keras.models.load_model(
    HEAD_MODEL_PATH
)

head_val_probs = predict_probabilities(
    head_model,
    val_cls_ds,
)

val_true = (
    val_df["label"]
    .astype(int)
    .values
)

head_val_auc = roc_auc_score(
    val_true,
    head_val_probs,
)

candidate_rows = [
    {
        "Model": (
            "Frozen EfficientNetB0 head"
        ),
        "Path": str(
            HEAD_MODEL_PATH
        ),
        "Validation ROC-AUC": (
            head_val_auc
        ),
    }
]

if (
    DO_FINE_TUNING
    and FINE_MODEL_PATH.exists()
):
    fine_model = (
        keras.models.load_model(
            FINE_MODEL_PATH
        )
    )

    fine_val_probs = (
        predict_probabilities(
            fine_model,
            val_cls_ds,
        )
    )

    fine_val_auc = roc_auc_score(
        val_true,
        fine_val_probs,
    )

    candidate_rows.append(
        {
            "Model": (
                "Fine-tuned EfficientNetB0"
            ),
            "Path": str(
                FINE_MODEL_PATH
            ),
            "Validation ROC-AUC": (
                fine_val_auc
            ),
        }
    )

classifier_comparison_df = (
    pd.DataFrame(
        candidate_rows
    )
)

classifier_comparison_df = (
    classifier_comparison_df
    .sort_values(
        "Validation ROC-AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    classifier_comparison_df
    .round(4)
)

SELECTED_CLASSIFIER_NAME = (
    classifier_comparison_df
    .loc[0, "Model"]
)

SELECTED_CLASSIFIER_PATH = (
    classifier_comparison_df
    .loc[0, "Path"]
)

final_classifier = (
    keras.models.load_model(
        SELECTED_CLASSIFIER_PATH
    )
)

print(
    "Selected classifier:",
    SELECTED_CLASSIFIER_NAME,
)
print(
    "Classification threshold:",
    CLS_THRESHOLD,
)

classifier_comparison_df.to_csv(
    RESULTS_DIR
    / "classifier_validation_comparison.csv",
    index=False,
)

## 25. Classification learning curves

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

head_epochs = range(
    1,
    len(
        head_history.history[
            "loss"
        ]
    ) + 1,
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "loss"
    ],
    label="Head train",
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "val_loss"
    ],
    label="Head val",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "auc"
    ],
    label="Head train",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "val_auc"
    ],
    label="Head val",
)

if fine_history is not None:
    offset = len(
        head_epochs
    )

    fine_epochs = range(
        offset + 1,
        offset
        + len(
            fine_history.history[
                "loss"
            ]
        )
        + 1,
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "loss"
        ],
        label="Fine train",
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "val_loss"
        ],
        label="Fine val",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "auc"
        ],
        label="Fine train",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "val_auc"
        ],
        label="Fine val",
    )

axes[0].set_title(
    "Classification loss"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title(
    "Classification ROC-AUC"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

# PART D — FINAL HELD-OUT TESTING

Only now are the 100 held-out test cases used for final performance reporting.

Final classification results include:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)
- sensitivity
- specificity
- accuracy
- balanced accuracy
- precision
- F1-score
- ROC-AUC

## 26. Final segmentation testing on the 100 held-out cases

In [ ]:
def evaluate_attention_segmentation(
    model,
    frame,
):
    rows = []

    tp_pixels = 0
    tn_pixels = 0
    fp_pixels = 0
    fn_pixels = 0

    for _, row in frame.iterrows():
        image_pil = load_rgb(
            row["image_path"]
        )

        true_mask_pil = load_mask(
            row["mask_path"]
        ).resize(
            (
                SEG_IMAGE_SIZE[1],
                SEG_IMAGE_SIZE[0],
            ),
            Image.Resampling.NEAREST,
        )

        true_mask = (
            np.asarray(true_mask_pil) > 127
        ).astype(np.uint8)

        _, pred_mask = predict_mask_for_pil(
            model,
            image_pil,
        )

        intersection = np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        true_sum = int(
            (true_mask == 1).sum()
        )
        pred_sum = int(
            (pred_mask == 1).sum()
        )

        union = np.logical_or(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        dice = (
            2 * intersection + EPS
        ) / (
            true_sum
            + pred_sum
            + EPS
        )

        iou = (
            intersection + EPS
        ) / (
            union + EPS
        )

        tp = np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()
        tn = np.logical_and(
            true_mask == 0,
            pred_mask == 0,
        ).sum()
        fp = np.logical_and(
            true_mask == 0,
            pred_mask == 1,
        ).sum()
        fn = np.logical_and(
            true_mask == 1,
            pred_mask == 0,
        ).sum()

        tp_pixels += tp
        tn_pixels += tn
        fp_pixels += fp
        fn_pixels += fn

        rows.append(
            {
                "image_id": row["image_id"],
                "diagnosis": row["diagnosis"],
                "image_path": row["image_path"],
                "mask_path": row["mask_path"],
                "dice": float(dice),
                "iou": float(iou),
            }
        )

    per_image_df = pd.DataFrame(rows)

    pixel_sensitivity = (
        tp_pixels
        / (
            tp_pixels
            + fn_pixels
            + EPS
        )
    )

    pixel_specificity = (
        tn_pixels
        / (
            tn_pixels
            + fp_pixels
            + EPS
        )
    )

    return (
        per_image_df,
        float(pixel_sensitivity),
        float(pixel_specificity),
    )


test_eval_dict = selected_segmenter.evaluate(
    test_seg_ds,
    verbose=0,
    return_dict=True,
)

(
    attention_test_per_image_df,
    attention_pixel_sensitivity,
    attention_pixel_specificity,
) = evaluate_attention_segmentation(
    selected_segmenter,
    test_df,
)

test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].mean()
)

test_jaccard = float(
    attention_test_per_image_df[
        "iou"
    ].mean()
)

test_loss = float(
    test_eval_dict["loss"]
)

best_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].max()
)

median_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].median()
)

worst_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].min()
)

attention_parameters = int(
    selected_segmenter.count_params()
)

attention_segmentation_results = pd.DataFrame(
    {
        "Metric": [
            "Test Dice",
            "Test Jaccard / IoU",
            "Test Loss",
            "Best Test Dice",
            "Median Test Dice",
            "Worst Test Dice",
            "Best Validation Dice",
            "Parameters",
            "Pixel Sensitivity",
            "Pixel Specificity",
        ],
        "Value": [
            test_dice,
            test_jaccard,
            test_loss,
            best_test_dice,
            median_test_dice,
            worst_test_dice,
            best_validation_dice,
            attention_parameters,
            attention_pixel_sensitivity,
            attention_pixel_specificity,
        ],
    }
)

print(
    "ATTENTION U-NET — FINAL HELD-OUT TEST RESULTS"
)
display(
    attention_segmentation_results
)

attention_test_per_image_df.to_csv(
    RESULTS_DIR
    / "attention_unet_test_per_image.csv",
    index=False,
)

attention_segmentation_results.to_csv(
    RESULTS_DIR
    / "attention_unet_final_test_results.csv",
    index=False,
)


### 26.1 Attention U-Net best, median and worst test segmentations

Each row shows the original dermoscopic image, expert ground-truth mask, predicted mask, and consistent red prediction overlay. Dice and IoU are calculated per image.


In [ ]:
sorted_seg = (
    attention_test_per_image_df
    .sort_values("dice")
    .reset_index(drop=True)
)

worst_row = sorted_seg.iloc[0]
best_row = sorted_seg.iloc[-1]

median_target = float(
    sorted_seg["dice"].median()
)
median_idx = (
    sorted_seg["dice"]
    .sub(median_target)
    .abs()
    .idxmin()
)
median_row = sorted_seg.loc[median_idx]

representative_rows = [
    ("Best", best_row),
    ("Median", median_row),
    ("Worst", worst_row),
]

fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 13),
)

for row_idx, (label, result_row) in enumerate(
    representative_rows
):
    image = load_rgb(
        result_row["image_path"]
    )

    gt_mask_pil = load_mask(
        result_row["mask_path"]
    ).resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0],
        ),
        Image.Resampling.NEAREST,
    )

    gt_mask = (
        np.asarray(gt_mask_pil) > 127
    ).astype(np.uint8)

    _, pred_mask = predict_mask_for_pil(
        selected_segmenter,
        image,
    )

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(
        f"{label} — Original\n"
        f"{result_row['image_id']}"
    )

    axes[row_idx, 1].imshow(
        gt_mask,
        cmap="gray",
    )
    axes[row_idx, 1].set_title(
        "Ground-Truth Mask"
    )

    axes[row_idx, 2].imshow(
        pred_mask,
        cmap="gray",
    )
    axes[row_idx, 2].set_title(
        "Predicted Mask"
    )

    axes[row_idx, 3].imshow(
        segmentation_overlay(
            image,
            pred_mask,
        )
    )
    axes[row_idx, 3].set_title(
        f"Prediction Overlay\n"
        f"Dice={result_row['dice']:.3f} | "
        f"IoU={result_row['iou']:.3f}"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("Best Dice:", round(float(best_row["dice"]), 4))
print("Median Dice:", round(float(median_row["dice"]), 4))
print("Worst Dice:", round(float(worst_row["dice"]), 4))

## 27. Final classification confusion matrix and clinical metrics

In [ ]:
test_true = (
    test_df["label"]
    .astype(int)
    .values
)

test_probs = (
    final_classifier
    .predict(
        test_cls_ds,
        verbose=0,
    )
    .reshape(-1)
)

test_pred = (
    test_probs
    >= CLS_THRESHOLD
).astype(int)

cm = confusion_matrix(
    test_true,
    test_pred,
    labels=[0, 1],
)

tn, fp, fn, tp = (
    cm.ravel()
)

sensitivity = (
    tp / (tp + fn)
    if (tp + fn)
    else np.nan
)

specificity = (
    tn / (tn + fp)
    if (tn + fp)
    else np.nan
)

accuracy = accuracy_score(
    test_true,
    test_pred,
)

balanced_accuracy = (
    balanced_accuracy_score(
        test_true,
        test_pred,
    )
)

precision = precision_score(
    test_true,
    test_pred,
    zero_division=0,
)

f1 = f1_score(
    test_true,
    test_pred,
    zero_division=0,
)

if len(
    np.unique(test_true)
) == 2:
    roc_auc = roc_auc_score(
        test_true,
        test_probs,
    )
else:
    roc_auc = np.nan

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "True Positive (TP)",
            "True Negative (TN)",
            "False Positive (FP)",
            "False Negative (FN)",
            "Sensitivity",
            "Specificity",
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "F1-score",
            "ROC-AUC",
        ],
        "Value": [
            tp,
            tn,
            fp,
            fn,
            sensitivity,
            specificity,
            accuracy,
            balanced_accuracy,
            precision,
            f1,
            roc_auc,
        ],
    }
)

display(
    final_metrics
)

print(
    "\nClassification report:"
)

print(
    classification_report(
        test_true,
        test_pred,
        target_names=[
            "Benign",
            "Malignant",
        ],
        zero_division=0,
    )
)

final_metrics.to_csv(
    RESULTS_DIR
    / "classification_final_test_metrics.csv",
    index=False,
)

test_predictions_df = (
    test_df[
        [
            "image_id",
            "diagnosis",
            "image_path",
            "mask_path",
            "roi_path",
        ]
    ]
    .copy()
)

test_predictions_df[
    "malignancy_probability"
] = test_probs

test_predictions_df[
    "predicted_label"
] = test_pred

test_predictions_df[
    "predicted_diagnosis"
] = np.where(
    test_pred == 1,
    "malignant",
    "benign",
)

test_predictions_df.to_csv(
    RESULTS_DIR
    / "final_test_predictions.csv",
    index=False,
)

## 28. Confusion matrix visualisation

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 5)
)

ax.imshow(
    cm
)

ax.set_xticks(
    [0, 1]
)
ax.set_yticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "Predicted Benign",
        "Predicted Melanoma",
    ]
)

ax.set_yticklabels(
    [
        "Actual Benign",
        "Actual Melanoma",
    ]
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=18,
        )

ax.set_title(
    f"Final Test Confusion Matrix (n={len(test_df)})"
)
ax.set_xlabel(
    "Model prediction"
)
ax.set_ylabel(
    "Ground truth"
)

plt.tight_layout()
plt.show()

print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print(
    f"Sensitivity = "
    f"{sensitivity:.3f}"
)
print(
    f"Specificity = "
    f"{specificity:.3f}"
)
print(
    f"Accuracy = "
    f"{accuracy:.3f}"
)

## 29. ROC curve

In [ ]:
if len(
    np.unique(test_true)
) == 2:
    fpr, tpr, thresholds = (
        roc_curve(
            test_true,
            test_probs,
        )
    )

    plt.figure(
        figsize=(6, 5)
    )

    plt.plot(
        fpr,
        tpr,
        label=(
            f"EfficientNetB0 "
            f"AUC = {roc_auc:.3f}"
        ),
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
    )

    plt.xlabel(
        "False Positive Rate"
    )
    plt.ylabel(
        "True Positive Rate / Sensitivity"
    )
    plt.title(
        f"ROC Curve — {len(test_df)} Held-Out Test Cases"
    )
    plt.legend()
    plt.show()
else:
    print(
        "ROC curve cannot be calculated "
        "because the test set contains "
        "only one class."
    )

## 30. How to interpret the confusion matrix

- **True Positive (TP):** actual melanoma correctly predicted as melanoma.
- **True Negative (TN):** actual benign lesion correctly predicted as benign.
- **False Positive (FP):** benign lesion incorrectly predicted as melanoma.
- **False Negative (FN):** melanoma incorrectly predicted as benign.

### Sensitivity

\[
Sensitivity = \frac{TP}{TP + FN}
\]

Sensitivity asks:

> Of all true melanoma cases, how many did the AI correctly detect?

### Specificity

\[
Specificity = \frac{TN}{TN + FP}
\]

Specificity asks:

> Of all true benign cases, how many did the AI correctly identify as benign?

False negatives are particularly important in melanoma screening because they represent malignant lesions the model failed to flag.

## 31. Display example TP / TN / FP / FN predictions

In [ ]:
def outcome_type(
    true_label,
    pred_label,
):
    if (
        true_label == 1
        and pred_label == 1
    ):
        return "TP"

    if (
        true_label == 0
        and pred_label == 0
    ):
        return "TN"

    if (
        true_label == 0
        and pred_label == 1
    ):
        return "FP"

    return "FN"


test_predictions_df[
    "outcome"
] = [
    outcome_type(
        true_label,
        pred_label,
    )
    for true_label, pred_label
    in zip(
        test_true,
        test_pred,
    )
]

examples = []

for outcome in [
    "TP",
    "TN",
    "FP",
    "FN",
]:
    subset = (
        test_predictions_df[
            test_predictions_df[
                "outcome"
            ] == outcome
        ]
    )

    if len(subset):
        examples.append(
            subset.iloc[0]
        )

if examples:
    fig, axes = plt.subplots(
        len(examples),
        2,
        figsize=(
            10,
            4 * len(examples),
        ),
    )

    if len(examples) == 1:
        axes = np.array(
            [axes]
        )

    for row_idx, row in enumerate(
        examples
    ):
        original = load_rgb(
            row["image_path"]
        )

        roi = load_rgb(
            row["roi_path"]
        )

        axes[
            row_idx,
            0,
        ].imshow(
            original
        )

        axes[
            row_idx,
            0,
        ].set_title(
            f"{row['outcome']} | "
            f"True: {row['diagnosis']}"
        )

        axes[
            row_idx,
            0,
        ].axis(
            "off"
        )

        axes[
            row_idx,
            1,
        ].imshow(
            roi
        )

        axes[
            row_idx,
            1,
        ].set_title(
            f"Pred: "
            f"{row['predicted_diagnosis']} | "
            f"P(malignant)="
            f"{row['malignancy_probability']:.3f}"
        )

        axes[
            row_idx,
            1,
        ].axis(
            "off"
        )

    plt.tight_layout()
    plt.show()
else:
    print(
        "No example outcomes available."
    )

# PART E — GRAD-CAM EXPLAINABILITY

Grad-CAM visualises spatial regions that most influenced the classifier output.

It does **not** prove clinical causality and should not be interpreted as a diagnostic explanation.

## 32. Grad-CAM helper

In [ ]:
def make_gradcam_heatmap(
    model,
    image_batch,
):
    backbone = model.get_layer(
        "efficientnetb0"
    )

    gap = model.get_layer(
        "gap"
    )

    dropout = model.get_layer(
        "dropout"
    )

    prediction_layer = model.get_layer(
        "prediction"
    )

    with tf.GradientTape() as tape:
        feature_maps = backbone(
            image_batch,
            training=False,
        )

        tape.watch(
            feature_maps
        )

        x = gap(
            feature_maps
        )

        x = dropout(
            x,
            training=False,
        )

        prediction = (
            prediction_layer(
                x
            )
        )

    gradients = tape.gradient(
        prediction,
        feature_maps,
    )

    pooled_gradients = (
        tf.reduce_mean(
            gradients,
            axis=(1, 2),
        )
    )

    feature_maps = (
        feature_maps[0]
    )

    pooled_gradients = (
        pooled_gradients[0]
    )

    heatmap = tf.reduce_sum(
        feature_maps
        * pooled_gradients,
        axis=-1,
    )

    heatmap = tf.maximum(
        heatmap,
        0,
    )

    max_value = tf.reduce_max(
        heatmap
    )

    heatmap = (
        heatmap
        / (
            max_value
            + 1e-8
        )
    )

    return heatmap.numpy()


def prepare_roi_for_classifier(
    roi_pil,
):
    resized = (
        roi_pil
        .convert("RGB")
        .resize(
            (
                CLS_IMAGE_SIZE[1],
                CLS_IMAGE_SIZE[0],
            ),
            Image.Resampling.BILINEAR,
        )
    )

    array = (
        np.asarray(resized)
        .astype("float32")
    )

    return (
        resized,
        array[None, ...],
    )


def heatmap_overlay(
    image_pil,
    heatmap,
    alpha=0.40,
):
    image = image_pil.convert(
        "RGB"
    )

    heatmap_resized = (
        Image.fromarray(
            np.uint8(
                np.clip(
                    heatmap,
                    0,
                    1,
                )
                * 255
            )
        )
        .resize(
            image.size,
            Image.Resampling.BILINEAR,
        )
    )

    heat = (
        np.asarray(
            heatmap_resized
        )
        .astype(
            np.float32
        )
        / 255.0
    )

    base = (
        np.asarray(image)
        .astype(
            np.float32
        )
        / 255.0
    )

    cmap = plt.get_cmap(
        "jet"
    )

    coloured = cmap(
        heat
    )[..., :3]

    overlay = np.clip(
        (
            (1 - alpha)
            * base
        )
        + (
            alpha
            * coloured
        ),
        0,
        1,
    )

    return Image.fromarray(
        np.uint8(
            overlay
            * 255
        )
    )


print(
    "Grad-CAM helper ready."
)

## 33. Grad-CAM examples

In [ ]:
gradcam_examples = []

for outcome in [
    "TP",
    "TN",
    "FP",
    "FN",
]:
    subset = (
        test_predictions_df[
            test_predictions_df[
                "outcome"
            ] == outcome
        ]
    )

    if len(subset):
        gradcam_examples.append(
            subset.iloc[0]
        )

for row in gradcam_examples:
    roi_pil = load_rgb(
        row["roi_path"]
    )

    (
        roi_resized,
        roi_batch,
    ) = prepare_roi_for_classifier(
        roi_pil
    )

    heatmap = make_gradcam_heatmap(
        final_classifier,
        tf.convert_to_tensor(
            roi_batch
        ),
    )

    overlay = heatmap_overlay(
        roi_resized,
        heatmap,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
    )

    axes[0].imshow(
        load_rgb(
            row["image_path"]
        )
    )
    axes[0].set_title(
        f"Original — "
        f"{row['outcome']}"
    )
    axes[0].axis(
        "off"
    )

    axes[1].imshow(
        roi_resized
    )
    axes[1].set_title(
        "Lesion ROI"
    )
    axes[1].axis(
        "off"
    )

    axes[2].imshow(
        overlay
    )
    axes[2].set_title(
        f"Grad-CAM | "
        f"P(malignant)="
        f"{row['malignancy_probability']:.3f}"
    )
    axes[2].axis(
        "off"
    )

    plt.tight_layout()
    plt.show()

# FINAL RESULTS DASHBOARD

This section consolidates the final Attention U-Net segmentation results, training curves, best/median/worst segmentation examples, classifier performance, confusion matrix, ROC-AUC and representative Grad-CAM outputs in one place.


In [ ]:
print("=" * 72)
print("FINAL ATTENTION U-NET SEGMENTATION RESULTS")
print("=" * 72)

final_segmentation_table = pd.DataFrame(
    {
        "Metric": [
            "Test Dice",
            "Test Jaccard / IoU",
            "Test Loss",
            "Best Test Dice",
            "Median Test Dice",
            "Worst Test Dice",
            "Best Validation Dice",
            "Parameters",
        ],
        "Value": [
            test_dice,
            test_jaccard,
            test_loss,
            best_test_dice,
            median_test_dice,
            worst_test_dice,
            best_validation_dice,
            attention_parameters,
        ],
    }
)

display(final_segmentation_table)

# Attention U-Net training curves.
history = attention_history
epochs = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
)

axes[0].plot(epochs, history["loss"], label="Train")
axes[0].plot(
    epochs,
    history["val_loss"],
    linestyle="--",
    label="Validation",
)
axes[0].set_title("Attention U-Net Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(
    epochs,
    history["dice_coefficient"],
    label="Train",
)
axes[1].plot(
    epochs,
    history["val_dice_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[1].set_title("Attention U-Net Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()

axes[2].plot(
    epochs,
    history["iou_coefficient"],
    label="Train",
)
axes[2].plot(
    epochs,
    history["val_iou_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[2].set_title("Attention U-Net IoU / Jaccard")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].legend()

plt.tight_layout()
plt.show()

# Best / Median / Worst segmentation images.
sorted_seg = (
    attention_test_per_image_df
    .sort_values("dice")
    .reset_index(drop=True)
)

worst_row = sorted_seg.iloc[0]
best_row = sorted_seg.iloc[-1]

median_target = float(sorted_seg["dice"].median())
median_idx = (
    sorted_seg["dice"]
    .sub(median_target)
    .abs()
    .idxmin()
)
median_row = sorted_seg.loc[median_idx]

representative_rows = [
    ("Best", best_row),
    ("Median", median_row),
    ("Worst", worst_row),
]

fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 13),
)

for row_idx, (label, result_row) in enumerate(
    representative_rows
):
    image = load_rgb(result_row["image_path"])

    gt_mask_pil = load_mask(
        result_row["mask_path"]
    ).resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0],
        ),
        Image.Resampling.NEAREST,
    )

    gt_mask = (
        np.asarray(gt_mask_pil) > 127
    ).astype(np.uint8)

    _, pred_mask = predict_mask_for_pil(
        selected_segmenter,
        image,
    )

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(
        f"{label} — Original\n"
        f"{result_row['image_id']}"
    )

    axes[row_idx, 1].imshow(gt_mask, cmap="gray")
    axes[row_idx, 1].set_title("Ground-Truth Mask")

    axes[row_idx, 2].imshow(pred_mask, cmap="gray")
    axes[row_idx, 2].set_title("Predicted Mask")

    axes[row_idx, 3].imshow(
        segmentation_overlay(image, pred_mask)
    )
    axes[row_idx, 3].set_title(
        f"Dice={result_row['dice']:.3f} | "
        f"IoU={result_row['iou']:.3f}"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("\n" + "=" * 72)
print("FINAL EFFICIENTNETB0 CLASSIFICATION RESULTS")
print("=" * 72)

classification_dashboard = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Sensitivity",
            "Specificity",
            "Test ROC-AUC",
            "Precision",
            "F1-score",
            "True Positive",
            "True Negative",
            "False Positive",
            "False Negative",
        ],
        "Value": [
            accuracy,
            sensitivity,
            specificity,
            roc_auc,
            precision,
            f1,
            tp,
            tn,
            fp,
            fn,
        ],
    }
)

display(classification_dashboard)

# Confusion matrix.
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(
    ["Predicted Benign", "Predicted Melanoma"]
)
ax.set_yticklabels(
    ["Actual Benign", "Actual Melanoma"]
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=18,
        )

ax.set_title(
    f"Final Test Confusion Matrix (n={len(test_df)})"
)
ax.set_xlabel("Model prediction")
ax.set_ylabel("Ground truth")

plt.tight_layout()
plt.show()

# ROC curve.
if len(np.unique(test_true)) == 2:
    fpr, tpr, _ = roc_curve(
        test_true,
        test_probs,
    )

    plt.figure(figsize=(6, 5))
    plt.plot(
        fpr,
        tpr,
        label=f"EfficientNetB0 AUC = {roc_auc:.3f}",
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title("Final Test ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 72)
print("REPRESENTATIVE GRAD-CAM RESULTS")
print("=" * 72)

dashboard_gradcam_rows = []

for outcome in ["TP", "TN", "FP", "FN"]:
    subset = test_predictions_df[
        test_predictions_df["outcome"] == outcome
    ]

    if len(subset):
        dashboard_gradcam_rows.append(
            subset.iloc[0]
        )

for row in dashboard_gradcam_rows:
    roi_pil = load_rgb(
        row["roi_path"]
    )

    roi_resized, roi_batch = (
        prepare_roi_for_classifier(
            roi_pil
        )
    )

    heatmap = make_gradcam_heatmap(
        final_classifier,
        tf.convert_to_tensor(
            roi_batch
        ),
    )

    overlay = heatmap_overlay(
        roi_resized,
        heatmap,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
    )

    axes[0].imshow(
        load_rgb(row["image_path"])
    )
    axes[0].set_title(
        f"Original — {row['outcome']}\n"
        f"True: {row['diagnosis']}"
    )

    axes[1].imshow(roi_resized)
    axes[1].set_title(
        f"Lesion ROI\n"
        f"Pred: {row['predicted_diagnosis']}"
    )

    axes[2].imshow(overlay)
    axes[2].set_title(
        f"Grad-CAM\n"
        f"P(malignant)="
        f"{row['malignancy_probability']:.3f}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

final_segmentation_table.to_csv(
    RESULTS_DIR
    / "FINAL_attention_unet_metrics.csv",
    index=False,
)

classification_dashboard.to_csv(
    RESULTS_DIR
    / "FINAL_classifier_metrics.csv",
    index=False,
)

print(
    "\nFinal result tables saved to:",
    RESULTS_DIR,
)

# 36. Dataset bias and project limitations

This notebook uses a 300-case scope (200 modelling + 100 held-out testing), but important limitations remain.

### Small dataset
Only 200 cases are available for model development. Deep models may overfit despite augmentation, early stopping and transfer learning.

### Class imbalance
Melanoma is less common than benign disease. The classifier therefore uses class weighting and reports sensitivity, specificity, F1 and ROC-AUC rather than accuracy alone.

### Skin-tone diversity
The ISIC 2016 Task 3B dataset does not provide reliable skin-tone labels for a proper skin-tone subgroup analysis.

Therefore:

- this notebook **does not invent skin-tone labels**,
- image brightness is **not** treated as a skin-tone label,
- artificially darkening images is **not** claimed to solve demographic bias.

Future work should validate the model using a more diverse dermoscopic dataset with expert-annotated skin-tone/demographic information.

### Imaging conditions
ISIC images are dermoscopic images. Performance on ordinary smartphone photographs or other camera systems cannot be assumed.

### Segmentation error propagation
A poor U-Net mask can generate a poor lesion ROI, which can affect downstream EfficientNet classification.

### Grad-CAM
Grad-CAM visualises areas influencing the network. It does not prove the clinical cause of melanoma.

# 37. References

1. Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC).* arXiv:1605.01397.

2. Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.

3. Oktay, O., et al. (2018). *Attention U-Net: Learning Where to Look for the Pancreas.* arXiv:1804.03999.

4. Tan, M., & Le, Q. V. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML.

5. Selvaraju, R. R., et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.* ICCV.

6. International Skin Imaging Collaboration (ISIC), **2016 Task 3B: Segmented Lesion Classification**.

# Final workflow summary

```text
300 matched image + mask + diagnosis cases
                    │
        ┌───────────┴───────────┐
        │                       │
200 MODELLING              100 FINAL TEST
        │
 160 train + 40 val
        │
        ▼
   Attention U-Net
        │
Best validation checkpoint
        │
Predicted lesion mask
        │
Lesion ROI
        │
EfficientNetB0 transfer learning
        │
Validation model selection
        │
        └───────────────────────► 100 FINAL TEST
                                  │
                                  ▼
                     Attention U-Net:
                     Test Dice / IoU / Loss
                     Best / Median / Worst Dice
                     Best validation Dice
                     Parameters
                                  │
                                  ▼
                     EfficientNetB0:
                     Confusion Matrix
                     TP / TN / FP / FN
                     Sensitivity
                     Specificity
                     Accuracy
                     F1
                     ROC-AUC
                                  │
                                  ▼
                              Grad-CAM
```

The web-app/deployment component has been intentionally removed from this notebook and can be added later as a separate deployment notebook.
